# Assigment: PyTorch Model Zoo

The [PyTorch "Model Zoo"](https://pytorch.org/vision/stable/models.html) provides a large number of pre-trained CNN models and vision [data sets](https://pytorch.org/vision/stable/datasets.html)...

In [3]:
#imports
import torch
import torchvision
import torchvision.transforms as transforms

In [4]:
#transform input data (image) to tensor
transform = transforms.Compose(
    [transforms.ToTensor(),
     transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

#set batch size
batch_size = 4

trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                        download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size,
                                          shuffle=True, num_workers=2)

testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                       download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size,
                                         shuffle=False, num_workers=2)

classes = ('plane', 'car', 'bird', 'cat',
           'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

100%|██████████| 170M/170M [00:13<00:00, 12.6MB/s]


## Assignment 1:
Load a "*ResNet18*" from the torchvision model zoo and train it for 10 epochs

In [3]:
import torchvision.models as models
import torch.nn as nn

model = models.resnet18(weights=None)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 10)  # Für CIFAR-10: 10 Klassen





In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)


criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.001, momentum=0.9)

In [5]:
for epoch in range(10):
    running_loss = 0.0
    for i, data in enumerate(trainloader, 0):
        inputs, labels = data[0].to(device), data[1].to(device)

        optimizer.zero_grad()

        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        if i % 100 == 99:
            print(f"[{epoch + 1}, {i + 1:5d}] loss: {running_loss / 100:.3f}")
            running_loss = 0.0

print("Finished Training")

[1,   100] loss: 2.842
[1,   200] loss: 2.708
[1,   300] loss: 2.533
[1,   400] loss: 2.485
[1,   500] loss: 2.356
[1,   600] loss: 2.355
[1,   700] loss: 2.380
[1,   800] loss: 2.337
[1,   900] loss: 2.363
[1,  1000] loss: 2.341
[1,  1100] loss: 2.432
[1,  1200] loss: 2.267
[1,  1300] loss: 2.131
[1,  1400] loss: 2.234
[1,  1500] loss: 2.337
[1,  1600] loss: 2.254
[1,  1700] loss: 2.194
[1,  1800] loss: 2.221
[1,  1900] loss: 2.173
[1,  2000] loss: 2.231
[1,  2100] loss: 2.219
[1,  2200] loss: 2.139
[1,  2300] loss: 2.171
[1,  2400] loss: 1.979
[1,  2500] loss: 2.026
[1,  2600] loss: 2.078
[1,  2700] loss: 1.981
[1,  2800] loss: 1.993
[1,  2900] loss: 2.082
[1,  3000] loss: 2.081
[1,  3100] loss: 2.094
[1,  3200] loss: 2.085
[1,  3300] loss: 2.014
[1,  3400] loss: 2.061
[1,  3500] loss: 2.025
[1,  3600] loss: 2.185
[1,  3700] loss: 2.011
[1,  3800] loss: 1.863
[1,  3900] loss: 2.047
[1,  4000] loss: 1.990
[1,  4100] loss: 1.978
[1,  4200] loss: 2.051
[1,  4300] loss: 1.856
[1,  4400] 

In [7]:
import torch
from torch.utils.data import DataLoader

def evaluate_accuracy(model, dataloader, device):
    model.eval()  # Modell in Evaluationsmodus setzen
    correct = 0
    total = 0

    with torch.no_grad():  # Keine Gradienten berechnen
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)  # Klassenvorhersage
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f'Accuracy on test set: {accuracy:.2f}%')
    return accuracy


In [7]:
evaluate_accuracy(model, testloader, device)


Accuracy on test set: 72.48%


72.48

## Assigment 2:
Load a **pre-trained** (on ImageNet) "*ResNet18*" from the torchvision model zoo and *fine-tune* it for ten epochs

In [1]:
from torchvision.models import resnet50, ResNet50_Weights


# New weights with accuracy 80.858%
model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth
100%|██████████| 97.8M/97.8M [00:00<00:00, 227MB/s]


In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision.models import resnet18, ResNet18_Weights
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Daten vorbereiten
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # ResNet erwartet 224x224
    transforms.ToTensor(),
])


# Vortrainiertes ResNet18 anpassen
model.fc = nn.Linear(model.fc.in_features, 10)  # letzte Schicht anpassen


# Loss und Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)

model = model.to(device)

# Training (10 Epochen)
print("Starte Training...")
for epoch in range(10):
    model.train()
    running_loss = 0.0
    for batch_idx, (inputs, labels) in enumerate(trainloader):
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        if (batch_idx + 1) % 100 == 0:
            print(f"Epoch {epoch+1}, Batch {batch_idx+1}, Loss: {loss.item():.4f}")

    avg_loss = running_loss / len(trainloader)
    print(f"==> Epoch {epoch+1} abgeschlossen, Durchschnittlicher Loss: {avg_loss:.4f}")




Starte Training...
Epoch 1, Batch 100, Loss: 2.6177
Epoch 1, Batch 200, Loss: 2.4223
Epoch 1, Batch 300, Loss: 2.3301
Epoch 1, Batch 400, Loss: 2.4369
Epoch 1, Batch 500, Loss: 2.1633
Epoch 1, Batch 600, Loss: 2.2607
Epoch 1, Batch 700, Loss: 2.2095
Epoch 1, Batch 800, Loss: 2.3374
Epoch 1, Batch 900, Loss: 2.2168
Epoch 1, Batch 1000, Loss: 2.1970
Epoch 1, Batch 1100, Loss: 2.3420
Epoch 1, Batch 1200, Loss: 2.2182
Epoch 1, Batch 1300, Loss: 2.6418
Epoch 1, Batch 1400, Loss: 1.9902
Epoch 1, Batch 1500, Loss: 2.3977
Epoch 1, Batch 1600, Loss: 2.2338
Epoch 1, Batch 1700, Loss: 2.2943
Epoch 1, Batch 1800, Loss: 2.2572
Epoch 1, Batch 1900, Loss: 2.2287
Epoch 1, Batch 2000, Loss: 2.2501
Epoch 1, Batch 2100, Loss: 2.3167
Epoch 1, Batch 2200, Loss: 2.2886
Epoch 1, Batch 2300, Loss: 2.1272
Epoch 1, Batch 2400, Loss: 2.3514
Epoch 1, Batch 2500, Loss: 2.2735
Epoch 1, Batch 2600, Loss: 2.2358
Epoch 1, Batch 2700, Loss: 2.4599
Epoch 1, Batch 2800, Loss: 2.2268
Epoch 1, Batch 2900, Loss: 2.1438
Epoc

In [8]:
evaluate_accuracy(model, testloader, device)

Accuracy on test set: 11.14%


11.14